# What is the Agent Governance Toolkit?

The **Agent Governance Toolkit (AGT)** puts a deterministic policy layer around an AI agent, so that
**every prompt and every tool call is checked before it runs** -- by ordinary code, not by another model.

This notebook is the 5-minute tour. It uses the **real** toolkit (`agent_os`), runs fully offline, and
proves four things:

>
> **Why it matters.** Asking a model nicely to "follow the rules" is a *request* to a system that is
> random by design. AGT makes the unsafe action **structurally impossible** in deterministic code.

| # | Section | What you'll see |
|---|---------|-----------------|
| 1 | Health check (`agt doctor`) | Every control subsystem imports and runs |
| 2 | How a decision is made | A policy is data; the verdict is deterministic |
| 3 | Sub-millisecond enforcement | 10,000 real policy evaluations, live, with latency percentiles |
| 4 | Zero-trust tool calls | One gate in front of every action -- safe passes, dangerous blocked |



In [2]:
import warnings, sys, time, logging
warnings.filterwarnings("ignore")
logging.getLogger("agent_os").setLevel(logging.CRITICAL)   # keep the demo output clean
if sys.platform == "win32":
    try: sys.stdout.reconfigure(encoding="utf-8")
    except Exception: pass
from IPython.display import HTML, display

def header(kicker, title, subtitle):
    display(HTML(f"""<div style="font-family:Segoe UI,system-ui,sans-serif;
      background:linear-gradient(135deg,#eef4ff,#e7fbf5);border:1px solid #cfe0fb;
      border-radius:14px;padding:18px 22px;margin:14px 0 8px">
      <div style="font:700 11px ui-sans-serif;letter-spacing:.16em;color:#5566aa;
        text-transform:uppercase">{kicker}</div>
      <div style="font:700 22px ui-sans-serif;color:#13203d;margin-top:4px">{title}</div>
      <div style="color:#3a4a72;margin-top:6px;font-size:14.5px">{subtitle}</div></div>"""))

def kv_table(rows, head=None):
    body = "".join(
        f"<tr><td style='padding:6px 12px;color:#3a4a72'>{k}</td>"
        f"<td style='padding:6px 12px;font-weight:600'>{v}</td></tr>" for k, v in rows)
    cap = f"<caption style='text-align:left;color:#5566aa;font:600 12px ui-sans-serif;padding:4px 0'>{head}</caption>" if head else ""
    display(HTML(f"<table style='border-collapse:collapse;font:13.5px ui-sans-serif'>{cap}{body}</table>"))

try:
    import agent_os, agent_compliance
    from importlib.metadata import version
    print("Agent Governance Toolkit is ready in this kernel.")
except ModuleNotFoundError:
    print("Not installed in this kernel.\n"
          "Run:  pip install agent-governance-toolkit\n"
          "then select that environment as the notebook kernel and re-run.")

Agent Governance Toolkit is ready in this kernel.


## 1 &middot; Health check (`agt doctor`)
Before you trust a governance layer, prove it is actually wired up. This is a real preflight you can run in CI.

In [3]:
header("Health check (agt doctor)", "Is the governance kernel wired up correctly?",
       "A real preflight: each control subsystem is imported and exercised once. Drop this into CI.")

import time
from importlib.metadata import version
from agent_os.health import HealthChecker, ComponentHealth, HealthStatus
from agent_os.policies import PolicyEvaluator
from agent_os.policies.schema import PolicyDocument, PolicyDefaults, PolicyAction
from agent_os.prompt_injection import PromptInjectionDetector
from agent_os.sandbox import ExecutionSandbox
from agent_os.circuit_breaker import CircuitBreaker
from agent_os.memory_guard import MemoryGuard
from agent_os.audit_logger import GovernanceAuditLogger
from agent_os.mcp_security import MCPSecurityScanner

_minimal = PolicyDocument(name="health", version="1.0", defaults=PolicyDefaults(action=PolicyAction.ALLOW), rules=[])
probes = {
    "Policy engine":      lambda: PolicyEvaluator(policies=[_minimal]).evaluate({"tool_name": "noop"}),
    "Prompt injection":   lambda: PromptInjectionDetector().detect("hello", source="health"),
    "Execution sandbox":  lambda: ExecutionSandbox().validate_code("x = 1"),
    "Circuit breaker":    lambda: CircuitBreaker(),
    "Memory guard":       lambda: MemoryGuard().validate_write("hello", source="health"),
    "Audit logger":       lambda: GovernanceAuditLogger(),
    "MCP scanner":        lambda: MCPSecurityScanner().scan_tool("noop", "a harmless tool"),
}

def make_check(name, probe):
    def run():
        t0 = time.perf_counter()
        probe()
        return ComponentHealth(name=name, status=HealthStatus.HEALTHY, message="ready",
                               latency_ms=(time.perf_counter() - t0) * 1000)
    return run

doctor = HealthChecker(version=version("agent-os-kernel"))
for name, probe in probes.items():
    doctor.register_check(name, make_check(name, probe))
report = doctor.check_health()

rows = "".join(
    f"<tr><td style='padding:6px 12px'>"
    f"<span style='background:#e9f7ef;color:#0a7a47;border:1px solid #bfe5cf;border-radius:999px;"
    f"padding:2px 9px;font:700 11px ui-sans-serif'>OK</span></td>"
    f"<td style='padding:6px 12px;font-family:ui-monospace'>{name}</td>"
    f"<td style='padding:6px 12px;color:#3a4a72'>{c.message}</td>"
    f"<td style='padding:6px 12px;color:#6b7da3'>{c.latency_ms:.2f} ms</td></tr>"
    for name, c in report.components.items())
display(HTML(
    f"<table style='border-collapse:collapse;font:13.5px ui-sans-serif;margin-top:8px'>"
    f"<tr style='color:#5566aa;text-align:left'><th style='padding:6px 12px'>Status</th>"
    f"<th style='padding:6px 12px'>Subsystem</th><th style='padding:6px 12px'>Check</th>"
    f"<th style='padding:6px 12px'>Latency</th></tr>{rows}</table>"
    f"<div style='margin-top:12px;font:600 14px ui-sans-serif;color:#0a7a47'>"
    f"Doctor result: {len(report.components)}/{len(report.components)} subsystems healthy "
    f"(kernel {version('agent-os-kernel')}, toolkit {version('agent-governance-toolkit')}).</div>"))

Status,Subsystem,Check,Latency
OK,Policy engine,ready,0.10 ms
OK,Prompt injection,ready,3.16 ms
OK,Execution sandbox,ready,0.11 ms
OK,Circuit breaker,ready,0.03 ms
OK,Memory guard,ready,0.71 ms
OK,Audit logger,ready,0.00 ms
OK,MCP scanner,ready,5.48 ms


## 2 &middot; How a decision is made
A policy is just **data** (an allow-list, some blocked patterns). The verdict is **deterministic code** -- same input, same answer, every time, with a human-readable reason.

In [5]:
header("How a decision is made", "A policy is data. The verdict is deterministic code.",
       "No model, no randomness. The same input always produces the same allow/deny with a reason you can read.")

from agent_os.integrations.base import PolicyInterceptor, GovernancePolicy, ToolCallRequest

# A least-privilege policy for a read-only reporting agent.
policy = GovernancePolicy(name="reporting-agent",
                          allowed_tools=["get_balance", "list_transactions"])
gate = PolicyInterceptor(policy)

for tool in ["get_balance", "transfer_funds"]:
    decision = gate.intercept(ToolCallRequest(tool_name=tool, arguments={}, agent_id="reporter"))
    mark = "ALLOW" if decision.allowed else "DENY"
    print(f"  {tool:<18} -> {mark:<6} {decision.reason or 'on the allow-list'}")

print("\nThe agent (and any prompt-injected version of it) can only ever call what the policy allows.")

  get_balance        -> ALLOW  on the allow-list
  transfer_funds     -> DENY   Tool 'transfer_funds' not in allowed list: ['get_balance', 'list_transactions']

The agent (and any prompt-injected version of it) can only ever call what the policy allows.


## 3 &middot; Sub-millisecond enforcement
Governance only works if it's cheap enough to run on *every* action. Here is the real cost, measured live.

In [6]:
header("Sub-millisecond enforcement", "Your LLM call: ~800 ms.  A governance check: a few microseconds.",
       "We build a 10-rule policy and evaluate it 10,000 times, live, then report real latency percentiles.")

from agent_os.policies import PolicyEvaluator
from agent_os.policies.schema import (PolicyDocument, PolicyRule, PolicyCondition,
                                      PolicyAction, PolicyOperator, PolicyDefaults)

rules = [PolicyRule(name=f"block-{t}",
                    condition=PolicyCondition(field="tool_name", operator=PolicyOperator.IN, value=[t]),
                    action=PolicyAction.DENY, priority=100, message=f"{t} blocked")
         for t in ["delete_file", "shell_exec", "execute_code", "drop_table",
                   "transfer_funds", "exfiltrate", "ssh_connect", "run_binary", "fork_bomb"]]
rules.append(PolicyRule(name="block-pii",
                        condition=PolicyCondition(field="input_text", operator=PolicyOperator.MATCHES,
                                                  value=r"\b\d{3}-\d{2}-\d{4}\b"),
                        action=PolicyAction.DENY, priority=90, message="PII (SSN) blocked"))
evaluator = PolicyEvaluator(policies=[PolicyDocument(name="prod", version="1.0",
                            defaults=PolicyDefaults(action=PolicyAction.ALLOW), rules=rules)])

allow_ctx = {"tool_name": "get_balance", "input_text": "show my balance"}
deny_ctx = {"tool_name": "shell_exec", "input_text": "rm -rf /"}
for _ in range(500):                       # warm up
    evaluator.evaluate(allow_ctx)

N = 10_000
samples = []
t0 = time.perf_counter()
for i in range(N):
    s = time.perf_counter_ns()
    evaluator.evaluate(allow_ctx if i % 2 else deny_ctx)
    samples.append(time.perf_counter_ns() - s)
wall = time.perf_counter() - t0
samples.sort()
us = lambda p: samples[int(N * p)] / 1000
p50, p95, p99 = us(0.50), us(0.95), us(0.99)
ops = N / wall

kv_table([
    ("Evaluations", f"{N:,}"),
    ("Wall-clock time", f"{wall*1000:,.0f} ms"),
    ("Throughput", f"{ops:,.0f} evaluations / second"),
    ("Latency p50", f"{p50:.2f} microseconds"),
    ("Latency p95", f"{p95:.2f} microseconds"),
    ("Latency p99", f"{p99:.2f} microseconds"),
], head="Live result")
print(f"\nA single LLM call is typically 500-2000 ms. At p50 = {p50:.2f} us, the governance check "
      f"adds about {p50/1000/800*100:.4f}% to an 800 ms call -- effectively free.")

Evaluations,"10,000"
Wall-clock time,246 ms
Throughput,"40,681 evaluations / second"
Latency p50,16.60 microseconds
Latency p95,53.60 microseconds
Latency p99,90.80 microseconds



A single LLM call is typically 500-2000 ms. At p50 = 16.60 us, the governance check adds about 0.0021% to an 800 ms call -- effectively free.


## 4 &middot; Zero-trust: every action is checked
The same interceptor sits in front of every tool the agent can call.

In [7]:
header("Zero-trust: every action is checked", "The same gate sits in front of every tool the agent calls.",
       "Safe tools pass. Dangerous ones are blocked with a reason -- before they run, every single time.")

from agent_os.integrations.base import PolicyInterceptor, GovernancePolicy, ToolCallRequest

policy = GovernancePolicy(name="bank-teller-agent",
                          allowed_tools=["get_balance", "list_transactions", "open_ticket"],
                          blocked_patterns=["ignore previous instructions", "rm -rf", "/etc/passwd"])
gate = PolicyInterceptor(policy)

calls = [
    ("get_balance",       {"account": "L-7791"}),
    ("list_transactions", {"account": "L-7791"}),
    ("transfer_funds",    {"amount": 50000, "to": "external"}),
    ("shell_exec",        {"cmd": "rm -rf /"}),
    ("open_ticket",       {"subject": "card lost"}),
]
rows = []
allowed = 0
for tool, args in calls:
    d = gate.intercept(ToolCallRequest(tool_name=tool, arguments=args, agent_id="teller"))
    allowed += d.allowed
    color = "#0a7a47" if d.allowed else "#c0334d"
    label = "ALLOW" if d.allowed else "BLOCK"
    rows.append(f"<tr><td style='padding:6px 12px;font-family:ui-monospace'>{tool}</td>"
                f"<td style='padding:6px 12px;color:{color};font-weight:700'>{label}</td>"
                f"<td style='padding:6px 12px;color:#3a4a72'>{d.reason or 'on the allow-list'}</td></tr>")
display(HTML(
    f"<table style='border-collapse:collapse;font:13px ui-sans-serif;margin-top:8px;width:100%'>"
    f"<tr style='color:#5566aa;text-align:left'><th style='padding:6px 12px'>Tool call</th>"
    f"<th style='padding:6px 12px'>Verdict</th><th style='padding:6px 12px'>Reason</th></tr>"
    f"{''.join(rows)}</table>"
    f"<div style='margin-top:10px;color:#3a4a72'><b style='color:#0a7a47'>{allowed} allowed</b> &middot; "
    f"<b style='color:#c0334d'>{len(calls)-allowed} blocked</b> &middot; "
    f"policy violations that reached execution: <b>0</b></div>"))

Tool call,Verdict,Reason
get_balance,ALLOW,on the allow-list
list_transactions,ALLOW,on the allow-list
transfer_funds,BLOCK,"Tool 'transfer_funds' not in allowed list: ['get_balance', 'list_transactions', 'open_ticket']"
shell_exec,BLOCK,"Tool 'shell_exec' not in allowed list: ['get_balance', 'list_transactions', 'open_ticket']"
open_ticket,ALLOW,on the allow-list


## Recap

In five minutes, with the real toolkit:

- **`agt doctor`** confirmed every control subsystem is healthy.
- A **policy is data**; the verdict is **deterministic** and explainable.
- Enforcement costs **microseconds** -- thousands of checks per millisecond.
- One **interceptor** governs every tool call; nothing unsafe reaches execution.